# Reaction-Diffusion System with Pattern Formation Demo
## Problem Setup
Consider a simply connected domain $\Omega$ with boundary $\partial \Omega$ and let $t \in I = [0,T]$. 
The model problem for the time-dependent PDE of the two-
component reaction-diffusion system reads

$$
\frac{\partial u}{\partial t} - D_u \Delta u - f_1(u,v) = 0 \, \text{in} \, x \in \Omega, \, t \in I, \\ 
\frac{\partial v}{\partial t} - D_v \Delta v - f_2(u,v) = 0 \, \text{in} \, x \in \Omega, \, t \in I,  
\\ 
\nabla u \cdot n = g_1, \, \nabla v \cdot n = g_2 \, \text{on} \, x \in \partial \Omega, \, t \in I, \\
u(x(0), 0) = u_0, \, v(x(0), 0) = v_0 \, \text{in} \, x \in \Omega.
$$

The source functions $f_1, f_2$ may vary with space and time.The initial conditions $u_0, v_0$ is a function of space only.

We apply the Schnakenberg model which has source functions
$$
f_1(u,v) = a - u + u^2 v, \\
f_2(u,v) = b - u^2v. 
$$ 

Together, we get the problem:
$$
\frac{\partial u}{\partial t} = D_u \Delta u + a - u + u^2 v \, \text{in} \, x \in \Omega, \, t \in I, \\ 
\frac{\partial v}{\partial t} = D_v \Delta v + b - u^2 v \, \text{in} \, x \in \Omega, \, t \in I.
$$

## Steady State solution of the Brusselator model
We can find the steady state solution by finding $(u_0, v_0)$ such that $f_1(u,v)=f_2(u,v)=0$. 
$$
a - u + u^2 v = 0, \\
b - u^2 v = 0.
$$
The second equation gives $b = u^2 v$ and substituting into the first equation yields $u = a + b$, consequently, $v = \frac{b}{(a+b)^2}$. The steady state solution is, then:
$$
(u_0, v_0) = \left(a+b, \frac{b}{(a+b)^2}\right).
$$

## The Variational Formulation

For the variational derivation, we highlight IMEX. 

IMEX (Implicit-Explicit) finite element method is a numerical technique that combines spatial descretization via finite elements with split implicit-explicit time integration. It treats stiff or linear terms implicitly for stability and non-stiff or nonlinear terms explicitly to save computational time. 

As such, for time step $t_{n+1}$, we have the discretization for $u$
$$ 
\left(\frac{\partial u}{\partial t}\right)^{n+1} - D_u \Delta u^{n+1} - f_1^{n} = 0.
$$

The only critical difference is that the source function is now defined explicitly, everything else remains the same. 

The rest of the variational formulation remains the same. 
The resulting weak formulation in bilinear form and linear form:
$$ 
a_u(u,w) = L_{u, n+1}(w),
$$
where 
$$
a_u(u,w) = \int_\Omega \left( uw + D_u \Delta t \nabla u \cdot \nabla w \right) dx \\
L_{u, n+1}(w) = \int_\Omega \left(u^n + \Delta t f_1^{n} \right) \cdot w dx - D_u \Delta t \int_{\partial \Omega} g_1 w ds.
$$

We notice that everything in the linear form is now known, it does not need to be solved implicitly. 

Similarly, for $v$ we have 
$$ 
a_v(v,w) = L_{v, n+1}(w),
$$
where 
$$
a_v(v,w) = \int_\Omega \left( vw + D_v \Delta t \nabla v \cdot \nabla w \right) dx \\
L_{v, n+1}(w) = \int_\Omega \left(v^n + \Delta t f_2^{n} \right) \cdot w dx - D_v \Delta t \int_{\partial \Omega} g_2 w ds.
$$

So for the Laplacian operator, we get 
$$
-\int_\Omega \left( \nabla^2 u\right)wdx = \int_\Omega \nabla u \cdot \nabla w dx + \int_{\partial \Omega} g w ds, 
$$

and the resulting weak formulation in bilinear form and linear form for $u$ and $v$ are:
$$ 
a_u(u,w) = L_{u, n+1}(w),
$$
where 
$$
a_u(u,w) = \int_\Omega \left( uw + D_u \Delta t \nabla u \cdot \nabla w \right) dx \\
L_{u, n+1}(w) = \int_\Omega \left(u^n + \Delta t f_1^{n+1} \right) \cdot w dx - D_u \Delta t \int_{\Lambda_N} g_1 w ds,
$$
and
$$ 
a_v(v,w) = L_{v, n+1}(w),
$$
where 
$$
a_v(v,w) = \int_\Omega \left( vw + D_v \Delta t \nabla v \cdot \nabla w \right) dx \\
L_{v, n+1}(w) = \int_\Omega \left(v^n + \Delta t f_2^{n+1} \right) \cdot w dx - D_v \Delta t \int_{\Lambda_N} g_2 w ds.
$$

## Parameters
For the FitzHugh-Nagumo model, we use parameters $D_u=1, D_v=10, \alpha = 0.01, \beta = 1$. We use $dx = 1$, $dt = 0.01$ and $T_f=10$. We have periodic boundaries for the left and right boundaries, and homogeneous Neumann boundaries for the top and bottom. 

## Implementation

Imports

In [1]:
from petsc4py import PETSc
from mpi4py import MPI 
import ufl 

from dolfinx import mesh, io, plot
from dolfinx.fem import (
    Constant,
    Function,
    functionspace,
    dirichletbc,
    form,
    locate_dofs_geometrical,
    extract_function_spaces,
)
from dolfinx.fem.petsc import (
    assemble_vector,
    assemble_matrix,
    create_vector,
)

import numpy as np 
import numpy.typing as npt

import matplotlib as mpl 
import pyvista 

Problem specific parameters

In [2]:
t = 0.0 # start time 
T = 10.0 # final time 
num_steps = 1000 # number of time steps 
dt = (T - t) / num_steps # time step size 
D_u = 1.0
a = 0.01
D_v = 10.0
b = 1

Lx = 20.0
Ly = 20.0 

Define the mesh and appropriate function spaces. 

In [3]:
nx, ny = 100, 100
domain = mesh.create_rectangle(
    MPI.COMM_WORLD, 
    [np.array([0, 0]), np.array([Lx, Ly])], 
    [nx, ny], 
    mesh.CellType.triangle
)
V = functionspace(domain, ("Lagrange", 1))

## Time-dependent output 
This is for visualizing in Paraview. 

In [4]:
xdmf_u = io.XDMFFile(domain.comm, "u.xdmf", "w")
xdmf_u.write_mesh(domain)

xdmf_v = io.XDMFFile(domain.comm, "v.xdmf", "w")
xdmf_v.write_mesh(domain)

## Define the initial conditions

In [5]:
u0 = a + b
v0 = b / (a + b)**2

rng = np.random.default_rng(seed=1)

def initial_condition(x, rng, u0):
    """Initial condition for the reaction-diffusion system."""
    u = u0 + 0.01 * rng.standard_normal(x.shape[1])
    return u

## Defining the variational formulation 

Having set $t=0$ in `u_exact`, we can reuse this variable to obtain $u_n$ for the initial time step. 

In [6]:
u_n = Function(V)
u_n.name = "u_n"
u_n.interpolate(lambda x: initial_condition(x, rng, u0))

v_n = Function(V)
v_n.name = "v_n"
v_n.interpolate(lambda x: initial_condition(x, rng, v0))

We implement $f_1, f_2$ as functions of $u$ and $v$. 

In [7]:
f_1 = a - u_n + u_n**2 * v_n
f_2 = b - u_n**2 * v_n

We can now create our variational formulation with the bilinear form `a_u`, `a_v` and the linear from `L_u`, `L_v`. 

We write the variational form on residual form, use `ufl.lhs()` and `ufl.rhs()` to extract the bilinear and linear forms. 

In [8]:
u = ufl.TrialFunction(V)
v = ufl.TrialFunction(V)
w_u = ufl.TestFunction(V)
w_v = ufl.TestFunction(V)

dx = ufl.dx(domain = domain)
ds = ufl.ds(domain = domain)

F_u = (
    u * w_u * dx
    + dt * D_u * ufl.dot(ufl.grad(u), ufl.grad(w_u)) * dx
    - (u_n + dt * f_1) * w_u * dx
)
a_u = form(ufl.lhs(F_u))
L_u = form(ufl.rhs(F_u))

F_v = (
    v * w_v * dx 
    + dt * D_v * ufl.dot(ufl.grad(v), ufl.grad(w_v)) * dx 
    - (v_n + dt * f_2) * w_v * dx
)
a_v = form(ufl.lhs(F_v))
L_v = form(ufl.rhs(F_v))

## Create the matrix and vector for the linear problem
For computational efficiency, we create structures that can reuse data, such as matrix sparsity patterns. 
Here, the bilinear form `a` is independent of time, so only need to assemble the matrix once. 

In [9]:
A_u = assemble_matrix(a_u)
A_u.assemble()
b_u = create_vector(extract_function_spaces(L_u))
uh = Function(V)
uh.name = "uh"
uh.x.array[:] = u_n.x.array
uh.x.scatter_forward()

A_v = assemble_matrix(a_v)
A_v.assemble()
b_v = create_vector(extract_function_spaces(L_v))
vh = Function(V)
vh.name = "vh"
vh.x.array[:] = v_n.x.array
vh.x.scatter_forward()

## Define a linear variational solver 
For the given problem we choose a direct solver and reuse the LU factorization at every time step. 

In [10]:
solver_u = PETSc.KSP().create(domain.comm)
solver_u.setOperators(A_u)
solver_u.setType(PETSc.KSP.Type.PREONLY)
pc_u = solver_u.getPC()
pc_u.setType(PETSc.PC.Type.LU)

solver_v = PETSc.KSP().create(domain.comm)
solver_v.setOperators(A_v)
solver_v.setType(PETSc.KSP.Type.PREONLY)
pc_v = solver_v.getPC()
pc_v.setType(PETSc.PC.Type.LU)

## Visualization of time dependent problem using pyvista 
We use the DOLFINx plotting functionality, based on `pyvista` to plot every time step. We visualize a colorbar reflecting the minimal and maximal value of $u$ and $v$ at each time step. 

In [11]:
grid_u = pyvista.UnstructuredGrid(*plot.vtk_mesh(V))

plotter_u = pyvista.Plotter()
plotter_u.open_gif("u_time.gif", fps=10)

grid_u.point_data["uh"] = uh.x.array
warped_u = grid_u.warp_by_scalar("uh", factor=1)

viridis = mpl.colormaps.get_cmap("viridis").resampled(25)
sargs = dict(
    title_font_size=25,
    label_font_size=20,
    fmt="%.2e",
    color="black",
    position_x=0.1,
    position_y=0.8,
    width=0.8,
    height=0.1,
)

renderer_u = plotter_u.add_mesh(
    warped_u,
    show_edges=True,
    lighting=False,
    cmap=viridis,
    scalar_bar_args=sargs,
    clim=[0, max(uh.x.array)],
)

grid_v = pyvista.UnstructuredGrid(*plot.vtk_mesh(V))

plotter_v = pyvista.Plotter()
plotter_v.open_gif("v_time.gif", fps=10)

grid_v.point_data["vh"] = vh.x.array
warped_v = grid_v.warp_by_scalar("vh", factor=1)

viridis = mpl.colormaps.get_cmap("viridis").resampled(25)
sargs = dict(
    title_font_size=25,
    label_font_size=20,
    fmt="%.2e",
    color="black",
    position_x=0.1,
    position_y=0.8,
    width=0.8,
    height=0.1,
)

renderer_v = plotter_v.add_mesh(
    warped_v,
    show_edges=True,
    lighting=False,
    cmap=viridis,
    scalar_bar_args=sargs,
    clim=[0, max(vh.x.array)],
)

2026-10-07 21:53:02.508 (   3.699s) [    7FE9D6C8B740]vtkXOpenGLRenderWindow.:1458  WARN| bad X server connection. DISPLAY=
libEGL warning: egl: failed to create dri2 screen
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied

2026-10-07 21:53:02.560 (   3.751s) [    7FE9D6C8B740] vtkEGLRenderWindow.cxx:359   WARN| vtkEGLRenderWindow (0x55f29ce539b0): EGL device index: 0 could not be initialized. Trying other devices...
libEGL warning: egl: failed to create dri2 screen
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied

2026-10-07 21:53:03.052 (   4.243s) [    7FE9D6C8B740] vtkEGLRenderWindow.cxx:359   WARN| vtkEGLRenderWindow (0x55f29ebdf6e0): EGL device index: 0 could not be initialized. Trying other devices...
libEGL warning: egl: failed to create dri2 screen
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Per

## Solve the time-dependent problem 
We can now create our time-stepping loop. 
In the loop, we update the Dirichlet boundary condition by interpolating the updated expression `u_exact` and `v_exact` into `V`. 
We reassemble the right hand side vector `b` with the updates `u_n` and `v_n` then apply the boundary conditions to these vectors. 
We do this using the lifting operation. 

Then we solve the problem using PETSc nad update `u_n` and `v_n` with the data from `uh` and `vh`. 

In [12]:
xdmf_u.write_function(uh, t)
xdmf_v.write_function(vh, t)

for n in range(num_steps):
    t += dt

    # Update the right hand side reusing the initial vector
    with b_u.localForm() as loc_b_u:
        loc_b_u.set(0)
    assemble_vector(b_u, L_u)

    with b_v.localForm() as loc_b_v:
        loc_b_v.set(0)
    assemble_vector(b_v, L_v)

    # Solve linear problem
    solver_u.solve(b_u, uh.x.petsc_vec)
    uh.x.scatter_forward()
    solver_v.solve(b_v, vh.x.petsc_vec)
    vh.x.scatter_forward()

    # Update solution at previous time step (u_n)
    u_n.x.array[:] = uh.x.array
    v_n.x.array[:] = vh.x.array

    # Write solution to file
    xdmf_u.write_function(uh, t)
    xdmf_v.write_function(vh, t)

    # Update plot 
    new_warped_u = grid_u.warp_by_scalar("uh", factor=1)
    warped_u.points[:, :] = new_warped_u.points
    warped_u.point_data["uh"][:] = uh.x.array
    plotter_u.write_frame()
    new_warped_v = grid_v.warp_by_scalar("vh", factor=1)
    warped_v.points[:, :] = new_warped_v.points
    warped_v.point_data["vh"][:] = vh.x.array
    plotter_v.write_frame()
plotter_u.close()
plotter_v.close()
xdmf_u.close()
xdmf_v.close()

libEGL warning: egl: failed to create dri2 screen
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied

2026-10-07 21:53:03.567 (   4.758s) [    7FE9D6C8B740] vtkEGLRenderWindow.cxx:359   WARN| vtkEGLRenderWindow (0x55f29ce539b0): EGL device index: 0 could not be initialized. Trying other devices...
libEGL warning: egl: failed to create dri2 screen
libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied

2026-10-07 21:53:03.892 (   5.083s) [    7FE9D6C8B740] vtkEGLRenderWindow.cxx:359   WARN| vtkEGLRenderWindow (0x55f29e6d01b0): EGL device index: 0 could not be initialized. Trying other devices...


Finally, free the PETSc object to avoid memory leaks. 

In [13]:
A_u.destroy()
b_u.destroy()
solver_u.destroy()

A_v.destroy()
b_v.destroy()
solver_v.destroy() 